# 01. Pipeline Configuration & Environment Initialization

This notebook centralizes all architectural configurations for the Medallion pipeline:
- **Catalogs**: `bronze_prod`, `silver_prod`, `gold_prod`
- **Bronze Schemas & Volumes**: `ecommerce` (raw landing & streaming checkpoints)
- **Silver Domain Schemas**: `customers`, `orders`, `products`, `region`
- **Gold Consumption Schemas**: `sales`, `customers`, `products`, `reference`
- **Audit & Hashing Utilities**: SHA-256 change detection and ETL audit stamps

Included in all downstream notebooks via `%run ./01_Config`


In [0]:
# Catalog and Schema Configuration Widgets
dbutils.widgets.text("bronze_catalog", "bronze_prod", "Bronze Catalog")
dbutils.widgets.text("silver_catalog", "silver_prod", "Silver Catalog")
dbutils.widgets.text("gold_catalog", "gold_prod", "Gold Catalog")
dbutils.widgets.text("source_schema", "ecommerce", "Source Schema")
dbutils.widgets.text("volume_name", "raw_landing", "Landing Volume")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
silver_catalog = dbutils.widgets.get("silver_catalog")
gold_catalog = dbutils.widgets.get("gold_catalog")
source_schema = dbutils.widgets.get("source_schema")
volume_name = dbutils.widgets.get("volume_name")


In [0]:
# Constants & Paths
BRONZE_CATALOG = bronze_catalog
SILVER_CATALOG = silver_catalog
GOLD_CATALOG = gold_catalog

BRONZE_SCHEMA = source_schema

# Silver Domain Schemas
SILVER_SCHEMAS = {
    "customers": "customers",
    "orders": "orders",
    "products": "products",
    "region": "region"
}

# Gold Consumption Schemas
GOLD_SCHEMAS = {
    "sales": "sales",
    "customers": "customers",
    "products": "products",
    "reference": "reference"
}

# Unity Catalog Volume Storage Locations
RAW_LANDING_VOLUME = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/{volume_name}"
CHECKPOINT_VOLUME = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/checkpoints"

# Spark Optimization Configurations
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

print(f"✓ Configured Bronze: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"✓ Configured Silver: {SILVER_CATALOG}.{{{', '.join(SILVER_SCHEMAS.values())}}}")
print(f"✓ Configured Gold:   {GOLD_CATALOG}.{{{', '.join(GOLD_SCHEMAS.values())}}}")
print(f"✓ Landing Volume:   {RAW_LANDING_VOLUME}")


In [0]:
def init_environment():
    """Initializes Unity Catalog catalogs, schemas, and volumes if they do not exist."""
    print("[INIT] Initializing Unity Catalog environment...")
    
    # 1. Create Catalogs
    for cat in [BRONZE_CATALOG, SILVER_CATALOG, GOLD_CATALOG]:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {cat}")
        print(f"  ✓ Catalog: {cat}")
        
    # 2. Create Bronze Source Schema & Volumes
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA} COMMENT 'Raw source data for {BRONZE_SCHEMA}'")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}.{volume_name}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}.checkpoints")
    print(f"  ✓ Bronze Schema & Volumes: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
    
    # 3. Create Silver Domain Schemas
    for schema_name in SILVER_SCHEMAS.values():
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.{schema_name} COMMENT 'Conformed domain: {schema_name}'")
        print(f"  ✓ Silver Schema: {SILVER_CATALOG}.{schema_name}")
        
    # 4. Create Gold Consumption Schemas
    for schema_name in GOLD_SCHEMAS.values():
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{schema_name} COMMENT 'Business consumption: {schema_name}'")
        print(f"  ✓ Gold Schema: {GOLD_CATALOG}.{schema_name}")
        
    print("[INIT] Unity Catalog initialization complete.")

# Execute environment setup
init_environment()


In [0]:
# Common Audit & Hashing Utilities
from pyspark.sql.functions import current_timestamp, lit, sha2, concat_ws, coalesce, col

def add_audit_columns(df, run_id=None):
    """Appends standard ETL audit metadata columns."""
    out = df.withColumn("_ingest_timestamp", current_timestamp())
    if run_id:
        out = out.withColumn("_run_id", lit(run_id))
    return out

def generate_record_hash(df, tracked_columns):
    """Computes SHA-256 hash across tracked business attributes for change detection."""
    return df.withColumn(
        "record_hash",
        sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("")) for c in tracked_columns]), 256)
    )

print("✓ Audit utilities and record_hash helper functions loaded.")
